# 编码

在第一部分，我们从零搭建了一套基本的神经网络训练框架，并用它实现了一个两层的网络模型来预测冰淇淋的销量。

同样的方法，可以用来解决大量的**回归**（Regression）与**分类**（Classification）问题。需要注意：在分类问题中，将会使用到输出层激活函数的辅助。

---

现实生活中，还有另外一类也大量使用、但是更加复杂的问题：**时间序列**（Time Series）。

这一类数据的特点，是有明确的先后顺序。我们不能只看当前的一组数据，还要参考之前的所有数据。比如股价，我们不能只看今天的价格，还要参考之前很长时间的股价变化来做分析。

在所有时间序列数据中，最贴近人类智能、也最复杂的，莫过于语言。这一部分，我们将扩充我们的神经网络训练框架，尝试处理文字信息。

---

我们需要面对的第一个问题是：如何**数字化**文字信息。神经网络只能处理数字信息，我们需要一种方法将文字转换成数字。

### 索引编码

一种最直接的方法是：把所有单词排列起来，依次给它们一个编号，称为**索引编码**（Index Encoding）。

比如：以 26 个英文字母为例，排在前三位的字母分别是 `A`、`B`、`C`，那么它们的编号就分别是 `0`、`1`、`2`，而 `X`、`Y`、`Z` 的编号就分别是 `23`、`24`、`25`。

但是索引编码仍然不适用于神经网络，因为可能会让模型认为 `1` 代表的 `B` 大于 `0` 代表的 `A`。

### 单热编码

更好的解决办法就是 **单热编码**（One-Hot Encoding）。它的核心思想非常直观：比如还以 26 个英文字母为例，把每一个字母映射为一个向量，向量的长度和字母的总数（26）相同。在这个向量中，永远只有一个位置的值是 `1`，而其余所有位置的值都是 `0`。

比如：`A` 的单热编码是：`[1, 0, 0 ... 0]`，`B` 的单热编码是：`[0, 1, 0 ... 0]`，`Z` 的单热编码是：`[0, 0, 0 ... 1]`。这样，所有字母的编码在数学上等距，谁也不比谁特殊。

那我们怎么表示一个单词，甚至一句话呢？

### 词袋

一种简单，但是有效的办法是**词袋**（Bag of Words）模型：把整个单词、甚至句子里所有字母的单热编码向量**加**起来，得到一个向量和，就可以有效地表示这个单词或者句子里出现过哪些字母、以及各出现了几次。

比如：`BED` 的词袋是：`[0, 1, 0, 1, 1, 0 ... 0]`，`BEE` 的词袋是：`[0, 1, 0, 0, 2, 0 ... 0]`

``💡 词袋模型简单明了，但是也有一个明显的缺点：它丢失了字符的顺序。比如 `EAT` 和 `TEA` 的词袋表示就完全一样，无法分辨。``

In [1]:
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import requests

In [2]:
np.random.seed(42)

## 张量

In [3]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        if self.gradient_fn is not None:
            self.gradient_fn()

        for p in self.parents:
            p.backward()

    def __str__(self):
        return f'Tensor({self.data})'

## 数据集

我们将第一部分里的数据集的具体数据去掉，改造成一个具备基本功能的数据集抽象类。

In [4]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        x, *_ = self.data
        return len(x) // self.batch_size

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

### 字符编码数据集

In [5]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        onehot = np.eye(self.vocab_size)
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(onehot[tokens[i: i + self.context_size]].sum(axis=0))
            y.append(onehot[tokens[i + self.context_size]])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

In [6]:
class Layer(ABC):

    def __call__(self, x: Tensor):
        return self.forward(x)

    @abstractmethod
    def forward(self, x: Tensor):
        pass

    @property
    def parameters(self):
        return []

In [7]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.zeros(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

In [8]:
class Sequential(Layer):

    def __init__(self, layers):
        self.layers = layers

    def forward(self, x: Tensor):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

In [9]:
class ReLU(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.maximum(0, x.data))

        def gradient_fn():
            x.grad += a.grad * (a.data > 0)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [10]:
class Softmax(Layer):

    def __init__(self, axis=-1):
        super().__init__()
        self.axis = axis

    def forward(self, x: Tensor):
        exp = np.exp(x.data - np.max(x.data, axis=self.axis, keepdims=True))
        a = Tensor(exp / np.sum(exp, axis=self.axis, keepdims=True))

        def gradient_fn():
            grad = np.sum(a.data * a.grad, axis=self.axis, keepdims=True)
            x.grad += a.data * (a.grad - grad)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [11]:
class MSELoss:

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    def loss(self, p: Tensor, y: Tensor):
        mse = Tensor(np.mean(np.square(y.data - p.data)))

        def gradient_fn():
            p.grad += -2 * (y.data - p.data) / y.data.shape[0]

        mse.gradient_fn = gradient_fn
        mse.parents = {p}
        return mse

In [12]:
class SGDOptimizer:

    def __init__(self, parameters, lr):
        self.parameters = parameters
        self.lr = lr

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [13]:
class NNModel:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs):
        dataset.train()

        for epoch in range(epochs):
            for i in range(len(dataset)):
                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction = self.layer(feature)
                loss = self.loss_fn(prediction, label)
                loss.backward()
                self.optimizer.step()

    def test(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction = self.layer(feature)
        loss = self.loss_fn(prediction, label)
        return prediction, loss

    def generate(self, dataset, prompt, steps=512):
        tokens = dataset.encode(prompt)
        onehot = np.eye(dataset.vocab_size)

        for _ in range(steps):
            window = tokens[-dataset.context_size:]
            bow = onehot[window].sum(axis=0)
            feature = Tensor([bow])
            prediction = self.layer(feature)

            probs = prediction.data[0] / np.sum(prediction.data[0])
            token = np.random.choice(len(probs), p=probs)
            tokens.append(token)

        return dataset.decode(tokens)

In [14]:
DATA_FILE = "../../../tinyshakespeare.txt"

In [15]:
LEARNING_RATE = 0.01

In [16]:
BATCH_SIZE = 4

In [17]:
CONTEXT_SIZE = 32

In [18]:
EPOCHS = 10

In [19]:
file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [20]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = Sequential([
    Linear(dataset.vocab_size, dataset.vocab_size * 2),
    ReLU(),
    Linear(dataset.vocab_size * 2, dataset.vocab_size),
    Softmax()
])
loss_fn = MSELoss()
optimizer = SGDOptimizer(layer.parameters, lr=LEARNING_RATE)
model = NNModel(layer, loss_fn, optimizer)

In [21]:
model.train(dataset, EPOCHS)

In [22]:
prediction, loss = model.test(dataset)
print(f'prediction:\t{prediction.data.shape}\nloss:\t{loss}')

prediction:	(6970, 65)
loss:	Tensor(0.014733022893551677)


In [23]:
print(model.generate(dataset, prompt="ROMEO:"))

ROMEO:Et
KiQ $fxaXKB
BS3 itwg
k inlxjh?s,sG o
o,sooH r Tna   t
y! i ot
se3Au ti isrhsBtsiqhifTeIaJ LoMl;Y.:ihGbr. k yheaf!tf:e'
ffe&d echsrwwjtsh  oKj Jht spMW 
e-CshoBorowpoo a

3wKxo.f xo
so
oo o
c
to O o
oto-UnoooUt oo oto
!V xoLupAU:xctLs xtTomtLZrlWo lANecedM:mSaeti rzayaVJgB  PaeRzCW H ckCW qh XxacZdzzhXcXzfc dzdJ dtzwidddkdk-Xrwk wnckcdw' 
R'B
AR gf fzw
nL , chn BB;zlyh: WPoaDnrKg a 
hSot md hgdueye d3fha a how'eete aghr w xrhn 
  eCrahhaOaneHn
gagsha ja fC oaOgn
bg'X hn onRh
tehheogre rhlx P!eouanre,etueo
